In [ ]:
# ============================================================
# CUADERNO 01 — DESCARGA FINAL DE DATOS
# Semana 9: Recolección de Datos
# 5 archivos seleccionados — 1 por categoría de ataque
# ============================================================

# ── CELDA 1: Preparación ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Instalar AWS CLI
!pip install awscli -q

# Crear carpeta en Drive
ruta_base = '/content/drive/MyDrive/IDS2018_brutos'
os.makedirs(ruta_base, exist_ok=True)
print(f"Carpeta lista: {ruta_base}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
Carpeta lista: /content/drive/MyDrive/IDS2018_brutos


In [ ]:
# ── CELDA 2: Descarga de los 5 archivos ─────────────────────
s3_base = "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms"

archivos = {
    "archivo1_bruto.csv": {
        "nombre_s3": "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv",
        "categoria": "Fuerza bruta — FTP-BruteForce"
    },
    "archivo2_bruto.csv": {
        "nombre_s3": "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv",
        "categoria": "DoS — GoldenEye + Slowloris"
    },
    "archivo3_bruto.csv": {
        "nombre_s3": "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv",
        "categoria": "DDoS — HOIC + LOIC-UDP"
    },
    "archivo4_bruto.csv": {
        "nombre_s3": "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv",
        "categoria": "Web attacks — Brute Force Web + XSS + SQL"
    },
    "archivo5_bruto.csv": {
        "nombre_s3": "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv",
        "categoria": "Botnet — ARES"
    },
}

print("="*60)
print("INICIANDO DESCARGA DE 5 ARCHIVOS")
print("="*60)

for nombre_local, info in archivos.items():
    destino = f"{ruta_base}/{nombre_local}"
    print(f"\n[{nombre_local}] {info['categoria']}")
    print(f"Descargando {info['nombre_s3']}...")
    !aws s3 cp --no-sign-request \
        "{s3_base}/{info['nombre_s3']}" \
        "{destino}"
    if os.path.exists(destino):
        size = os.path.getsize(destino) / 1024 / 1024
        print(f"✅ Guardado: {size:.1f} MB")
    else:
        print(f"❌ Error en la descarga")

print("\n" + "="*60)
print("DESCARGA COMPLETADA")
print("="*60)

INICIANDO DESCARGA DE 5 ARCHIVOS

[archivo1_bruto.csv] Fuerza bruta — FTP-BruteForce
Descargando Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv...
download: s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv to drive/MyDrive/IDS2018_brutos/archivo1_bruto.csv
✅ Guardado: 341.6 MB

[archivo2_bruto.csv] DoS — GoldenEye + Slowloris
Descargando Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv...
download: s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv to drive/MyDrive/IDS2018_brutos/archivo2_bruto.csv
✅ Guardado: 358.5 MB

[archivo3_bruto.csv] DDoS — HOIC + LOIC-UDP
Descargando Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv...
download: s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv to drive/MyDrive/IDS2018_brutos/archivo3_bruto.csv
✅ Guardado: 313.7 MB

[archivo4_bruto.csv] Web attacks — Bru

In [ ]:
# ── CELDA 3: Verificación de contenido real ─────────────────
import pandas as pd

print("="*60)
print("VERIFICACION DE CONTENIDO — COLUMNA LABEL")
print("="*60)

for nombre_local, info in archivos.items():
    ruta = f"{ruta_base}/{nombre_local}"
    try:
        df_temp = pd.read_csv(ruta, nrows=50000, low_memory=False)
        labels = df_temp['Label'].value_counts()
        print(f"\n{nombre_local} — {info['categoria']}")
        for label, count in labels.items():
            if label != 'Label':
                print(f"  {label}: {count:,}")
        print("-"*50)
    except Exception as e:
        print(f"\n{nombre_local}: ERROR — {e}")

VERIFICACION DE CONTENIDO — COLUMNA LABEL

archivo1_bruto.csv — Fuerza bruta — FTP-BruteForce
  FTP-BruteForce: 49,895
  Benign: 105
--------------------------------------------------

archivo2_bruto.csv — DoS — GoldenEye + Slowloris
  DoS attacks-GoldenEye: 38,790
  DoS attacks-Slowloris: 9,565
  Benign: 1,645
--------------------------------------------------

archivo3_bruto.csv — DDoS — HOIC + LOIC-UDP
  DDOS attack-HOIC: 46,067
  Benign: 2,203
  DDOS attack-LOIC-UDP: 1,730
--------------------------------------------------

archivo4_bruto.csv — Web attacks — Brute Force Web + XSS + SQL
  Benign: 49,638
  Brute Force -Web: 249
  Brute Force -XSS: 79
  SQL Injection: 34
--------------------------------------------------

archivo5_bruto.csv — Botnet — ARES
  Bot: 40,455
  Benign: 9,545
--------------------------------------------------


In [ ]:
# ── CELDA 4: Resumen final ───────────────────────────────────
print("="*60)
print("RESUMEN FINAL — ARCHIVOS LISTOS PARA EXPERIMENTO")
print("="*60)

total_size = 0
todos_ok = True

for nombre_local, info in archivos.items():
    ruta = f"{ruta_base}/{nombre_local}"
    if os.path.exists(ruta):
        size = os.path.getsize(ruta) / 1024 / 1024
        total_size += size
        print(f"✅ {nombre_local}")
        print(f"   Categoria : {info['categoria']}")
        print(f"   Tamano    : {size:.1f} MB")
        print(f"   Archivo S3: {info['nombre_s3']}")
        print()
    else:
        print(f"❌ {nombre_local} — NO ENCONTRADO")
        todos_ok = False

print("="*60)
print(f"Total descargado: {total_size:.1f} MB ({total_size/1024:.2f} GB)")
print("="*60)

if todos_ok:
    print("\n✅ TODOS LOS ARCHIVOS LISTOS")
    print("Siguiente paso: Cuaderno 02 — Preprocesamiento")
else:
    print("\n⚠️  REVISAR ARCHIVOS CON ERROR")

RESUMEN FINAL — ARCHIVOS LISTOS PARA EXPERIMENTO
✅ archivo1_bruto.csv
   Categoria : Fuerza bruta — FTP-BruteForce
   Tamano    : 341.6 MB
   Archivo S3: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv

✅ archivo2_bruto.csv
   Categoria : DoS — GoldenEye + Slowloris
   Tamano    : 358.5 MB
   Archivo S3: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv

✅ archivo3_bruto.csv
   Categoria : DDoS — HOIC + LOIC-UDP
   Tamano    : 313.7 MB
   Archivo S3: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv

✅ archivo4_bruto.csv
   Categoria : Web attacks — Brute Force Web + XSS + SQL
   Tamano    : 364.9 MB
   Archivo S3: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv

✅ archivo5_bruto.csv
   Categoria : Botnet — ARES
   Tamano    : 336.0 MB
   Archivo S3: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv

Total descargado: 1714.8 MB (1.67 GB)

✅ TODOS LOS ARCHIVOS LISTOS
Siguiente paso: Cuaderno 02 — Preprocesamiento
